In [8]:
from azure.ai.ml import MLClient, automl, Input
from azure.ai.ml.entities import Model, ManagedOnlineEndpoint
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import mlflow
import os

In [2]:
ml_client =ml_client = MLClient.from_config(credential=DefaultAzureCredential())
mlflow_client = mlflow.tracking.MlflowClient()

Found the config file in: /config.json


In [45]:
# Registered Azure ML data asset
training_data =Input(
        type="mltable",
        path="azureml:transaction-train-data:1"
)
validation_data = Input(
        type="mltable",
        path="azureml:transaction-val-data:1"
)
test_data = Input(
        type="mltable",
        path="azureml:transaction-test-data:1"
)
TARGET_COLUMN = "is_fraud"

# Create the AutoML classification job
fraud_detection_job = automl.classification(
    training_data=training_data,
    validation_data=validation_data,
    test_data=test_data,
    target_column_name=TARGET_COLUMN,
    primary_metric="norm_macro_recall",
    experiment_name="Default",
    compute="durrain20031",
    enable_model_explainability=True,

)
fraud_detection_job.set_limits(
    max_trials=6,
    max_concurrent_trials=2,
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(fraud_detection_job)
job_name = returned_job.name
print(f"Submitted job: {job_name}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Submitted job: gentle_dinner_4jdfgsnt12


In [3]:
job_name = 'gentle_dinner_4jdfgsnt12'

In [4]:
job = ml_client.jobs.get(job_name)
job

Experiment,Name,Type,Status,Details Page
Default,gentle_dinner_4jdfgsnt12,automl,Completed,Link to Azure Machine Learning studio


In [5]:
best_child_run_id = mlflow.get_run(job_name).data.tags['model_explain_best_run_child_id']
best_run = mlflow.get_run(best_child_run_id)

local_path = f"{os.path.dirname(os.getcwd())}/mlflow_artifacts"

# Download run's artifacts/outputs
local_path = mlflow_client.download_artifacts(
    best_run.info.run_id, "outputs", local_path
)
print(f"Artifacts downloaded in: {local_path}")
print(f"Artifacts: {os.listdir(local_path)}")

Artifacts downloaded in: /mnt/batch/tasks/shared/LS_root/mounts/clusters/durrain20031/code/FinRisk360/azureml/mlflow_artifacts/outputs
Artifacts: ['conda_env_v_1_0_0.yml', 'engineered_feature_names.json', 'env_dependencies.json', 'featurization_summary.json', 'generated_code', 'mlflow-model', 'model.pkl', 'pipeline_graph.json', 'run_id.txt', 'scoring_file_pbi_v_1_0_0.py', 'scoring_file_v_1_0_0.py', 'scoring_file_v_2_0_0.py']


In [7]:
model_name = "transaction-fraud-detection-model"
model = Model(
    path=f"azureml://jobs/{best_run.info.run_id}/outputs/artifacts/outputs/mlflow-model/",
    name=model_name,
    description="my sample object detection model",
    type=AssetTypes.MLFLOW_MODEL,
)

registered_model = ml_client.models.create_or_update(model)

In [26]:
# Creating a unique endpoint name with current datetime to avoid conflicts
import datetime

online_endpoint_name = "fraud-detection-" + datetime.datetime.now().strftime(
    "%M%f"
)

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="this is a sample online endpoint for deploying model",
    auth_mode="key",
    tags={"foo": "bar"},
)
print(online_endpoint_name)

fraud-detection-22654607


In [27]:
ml_client.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://fraud-detection-22654607.centralindia.inference.ml.azure.com/score', 'openapi_uri': 'https://fraud-detection-22654607.centralindia.inference.ml.azure.com/swagger.json', 'name': 'fraud-detection-22654607', 'description': 'this is a sample online endpoint for deploying model', 'tags': {'foo': 'bar'}, 'properties': {'createdBy': 'Durrain khan', 'createdAt': '2026-09-28T08:22:55.047588+0000', 'lastModifiedAt': '2026-09-28T08:24:21.982815+0000', 'azureml.onlineendpointid': '/subscriptions/9a8215b4-3616-4b88-a09f-88b52becafb8/resourcegroups/rg-ml-project-dev/providers/microsoft.machinelearningservices/workspaces/finrisk360/onlineendpoints/fraud-detection-22654607', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/9a8215b4-3616-4b88-a09f-88b52becafb8/providers/Microsoft.MachineLearningServices/locations/centralindia/mfeOperationsStatus/oeidp:158438c9-9d7b-

In [ ]:
response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    request_file="sample_request.json",
)

print(response)

In [ ]:
ml_client.online_endpoints.list